<a href="https://colab.research.google.com/github/shivanilokh/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivanilokh/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Abstract

Content teams need a practical way to identify which pages may be experiencing declining search performance so that limited review effort can be prioritized. This case study uses the FlyRank ML Internship anonymized content-refresh dataset to identify content with measurable signs of declining search performance and rank it for review. The workflow aggregates content-level search-performance data, defines a declining-performance label, creates pre-period features, trains a Random Forest model, and compares its ranking performance with a transparent rule-based baseline using Precision@50. The resulting ranked queue is intended as decision support for content and SEO review, helping teams prioritize which content to investigate first. The findings are directional and specific to the available dataset and evaluation setup; the model is not intended to predict Google's search algorithm or replace human review.

## Introduction

Declining search performance can create a content-maintenance problem: when many pages need attention, a team needs a practical way to decide which pages should be reviewed first. This project addresses that FlyRank content problem by treating the task as a ranking and prioritization problem rather than trying to predict search-engine behavior.

The analysis uses anonymized content-level search-performance data from the FlyRank ML Internship dataset. February performance signals are used to create features, while a later-period change in impressions is used to identify content showing a measurable decline. A simple rule-based baseline provides an interpretable reference point, and a Random Forest model is then used to rank content by its estimated likelihood of decline.

The final output is a ranked review queue that can help content and SEO teams focus their attention on higher-priority content first. The evaluation compares the machine-learning approach with the baseline using Precision@50. The result should be interpreted as directional evidence from this dataset and validation setup, with human review remaining necessary before taking content actions.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Can we identify content that shows measurable signs of declining search performance and rank it for review?

This analysis supports a practical content-maintenance decision: which content should be reviewed or improved first based on observed search-performance signals.


In [3]:
print("Capstone research question defined.")

Capstone research question defined.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data

This analysis uses the FlyRank ML Internship anonymized content-refresh dataset. The analysis uses the available content-level records and public-safe performance fields in the release. The dataset contains 30,000 content records and 45 columns covering content, search-performance, update, and trend-related signals.

Records were analyzed at the content level. Client identifiers were used only for grouped validation and were not used as model features. Target-derived fields such as `trend_direction` and `trend_pct` were excluded from the model features to avoid leakage. No client names, private queries, domains, credentials, or raw private exports are included in this analysis.


In [7]:
!pip -q install duckdb fsspec huggingface_hub

import duckdb
import pandas as pd
import numpy as np
import getpass
from huggingface_hub import login

token = getpass.getpass("Enter your Hugging Face READ token: ")
login(token=token)

con = duckdb.connect()

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_FEB = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
FACT_MAR = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

con.execute("SET VARIABLE hf_token = ?", [token])

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

print("Hugging Face connected.")
print("DuckDB ready.")

Enter your Hugging Face READ token: ··········
Hugging Face connected.
DuckDB ready.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Heading — Connect & Load Data



In [ ]:
# ============================================================
# CAPSTONE STEP 1 — CONNECT TO HUGGING FACE AND LOAD DATA
# ============================================================

!pip -q install duckdb fsspec huggingface_hub scikit-learn matplotlib

import duckdb
import pandas as pd
import numpy as np
import getpass
import matplotlib.pyplot as plt

from huggingface_hub import login

# Hugging Face login
token = getpass.getpass("Enter your Hugging Face READ token: ")
login(token=token)

# DuckDB connection
con = duckdb.connect()

# Dataset location
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_FEB = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
FACT_MAR = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# Give DuckDB access to Hugging Face
con.execute("SET VARIABLE hf_token = ?", [token])

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

print("✅ Hugging Face connected.")
print("✅ DuckDB connected.")
print("✅ Dataset paths configured.")

## 4. Heading — Load February & March Data


In [ ]:
# ============================================================
# CAPSTONE STEP 2 — LOAD FEBRUARY AND MARCH DATA
# ============================================================

feb = con.execute(f"""
    SELECT *
    FROM read_parquet('{FACT_FEB}')
""").df()

mar = con.execute(f"""
    SELECT *
    FROM read_parquet('{FACT_MAR}')
""").df()

print("✅ February rows:", len(feb))
print("✅ March rows:", len(mar))

print("\nColumns:")
print(feb.columns.tolist())

## 5. Heading — Create Content-Level Dataset



In [ ]:
# ============================================================
# CAPSTONE STEP 3 — CREATE CONTENT-LEVEL DATASET
# ============================================================

group_cols = ["client_hash_id", "content_hash_id"]

# February aggregation
feb_agg = feb.groupby(
    group_cols,
    as_index=False
).agg({
    "gsc_impressions": "sum",
    "gsc_clicks": "sum",
    "gsc_avg_position": "mean",
    "ga4_pageviews": "sum",
    "ga4_sessions": "sum",
    "sessions_organic": "sum",
    "scroll_events": "sum"
})

# March aggregation
mar_agg = mar.groupby(
    group_cols,
    as_index=False
).agg({
    "gsc_impressions": "sum",
    "gsc_clicks": "sum",
    "gsc_avg_position": "mean",
    "ga4_pageviews": "sum",
    "ga4_sessions": "sum",
    "sessions_organic": "sum",
    "scroll_events": "sum"
})

# Rename columns
feb_agg = feb_agg.rename(columns={
    "gsc_impressions": "feb_impressions",
    "gsc_clicks": "feb_clicks",
    "gsc_avg_position": "feb_position",
    "ga4_pageviews": "feb_pageviews",
    "ga4_sessions": "feb_sessions",
    "sessions_organic": "feb_organic",
    "scroll_events": "feb_scroll"
})

mar_agg = mar_agg.rename(columns={
    "gsc_impressions": "mar_impressions",
    "gsc_clicks": "mar_clicks",
    "gsc_avg_position": "mar_position",
    "ga4_pageviews": "mar_pageviews",
    "ga4_sessions": "mar_sessions",
    "sessions_organic": "mar_organic",
    "scroll_events": "mar_scroll"
})

# Merge February and March
df = feb_agg.merge(
    mar_agg,
    on=group_cols,
    how="inner"
)

print("✅ Modeling dataset created.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

## 6. Heading — Define Declining Content



In [ ]:
# ============================================================
# CAPSTONE STEP 4 — DEFINE DECLINING CONTENT
# ============================================================

# Calculate percentage change in impressions
df["impression_change_pct"] = np.where(
    df["feb_impressions"] > 0,
    (
        (df["mar_impressions"] - df["feb_impressions"])
        / df["feb_impressions"]
    ) * 100,
    0
)

# Declining = impressions decreased by 20% or more
df["is_declining_label"] = (
    df["impression_change_pct"] <= -20
).astype(int)

print("✅ Declining-performance label created.")

print("\nLabel counts:")
print(df["is_declining_label"].value_counts())

print("\nLabel percentages:")
print(
    df["is_declining_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

## 7. Heading — Create Features



In [ ]:
# ============================================================
# CAPSTONE STEP 5 — CREATE MODEL FEATURES
# ============================================================

# CTR for February
df["feb_ctr"] = np.where(
    df["feb_impressions"] > 0,
    df["feb_clicks"] / df["feb_impressions"],
    0
)

# CTR for March
df["mar_ctr"] = np.where(
    df["mar_impressions"] > 0,
    df["mar_clicks"] / df["mar_impressions"],
    0
)

# Change in position
df["position_change"] = (
    df["mar_position"] - df["feb_position"]
)

# Change in clicks
df["click_change"] = (
    df["mar_clicks"] - df["feb_clicks"]
)

# Change in pageviews
df["pageview_change"] = (
    df["mar_pageviews"] - df["feb_pageviews"]
)

# Change in organic sessions
df["organic_change"] = (
    df["mar_organic"] - df["feb_organic"]
)

print("✅ Features created.")

display(
    df[
        [
            "feb_impressions",
            "feb_clicks",
            "feb_ctr",
            "position_change",
            "click_change",
            "pageview_change",
            "organic_change",
            "is_declining_label"
        ]
    ].head()
)

## 8. Heading — Client-Level Train/Test Split


In [ ]:
# ============================================================
# CAPSTONE STEP 6 — CLIENT-LEVEL TRAIN/TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

clients = df["client_hash_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = df[
    df["client_hash_id"].isin(train_clients)
].copy()

test_df = df[
    df["client_hash_id"].isin(test_clients)
].copy()

print("✅ Client-level split completed.")

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

9. Heading — Train Random Forest

In [ ]:
# ============================================================
# CAPSTONE STEP 7 — TRAIN RANDOM FOREST
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Features available BEFORE the label period
feature_cols = [
    "feb_impressions",
    "feb_clicks",
    "feb_position",
    "feb_pageviews",
    "feb_sessions",
    "feb_organic",
    "feb_scroll",
    "feb_ctr"
]

X_train = train_df[feature_cols]
y_train = train_df["is_declining_label"]

X_test = test_df[feature_cols]
y_test = test_df["is_declining_label"]

# Handle missing values
imputer = SimpleImputer(strategy="median")

X_train_ready = imputer.fit_transform(X_train)
X_test_ready = imputer.transform(X_test)

# Random Forest
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(
    X_train_ready,
    y_train
)

print("✅ Random Forest trained successfully.")

10 Heading — Model Evaluation

In [ ]:
# ============================================================
# CAPSTONE STEP 8 — MODEL RANKING AND PRECISION@50
# ============================================================

# Predict decline probability
test_proba = model.predict_proba(
    X_test_ready
)[:, 1]

# Create ranking table
test_results = test_df[
    [
        "content_hash_id",
        "client_hash_id",
        "is_declining_label"
    ]
].copy()

test_results["decline_probability"] = test_proba

# Highest probability first
test_results = test_results.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

# Precision@50
top_50 = test_results.head(50)

model_precision_at_50 = (
    top_50["is_declining_label"].mean()
)

print(
    "✅ Random Forest Precision@50:",
    round(model_precision_at_50, 3)
)

print("\nTop 10 ranked content:")
display(test_results.head(10))

11  Heading — Rule-Based Baseline

In [ ]:
# ============================================================
# CAPSTONE STEP 9 — RULE-BASED BASELINE
# ============================================================

baseline_test = test_df.copy()

# Simple Week-4 style rules
baseline_test["baseline_score"] = (
    (baseline_test["feb_impressions"] >= 5000).astype(int) * 2
    +
    (baseline_test["feb_ctr"] < 0.05).astype(int)
    +
    (baseline_test["feb_position"] > 10).astype(int)
)

baseline_test = baseline_test.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_precision_at_50 = (
    baseline_test.head(50)["is_declining_label"].mean()
)

print(
    "✅ Rule-Based Baseline Precision@50:",
    round(baseline_precision_at_50, 3)
)

12 Heading — Model vs Baseline

In [ ]:
# ============================================================
# CAPSTONE STEP 10 — MODEL VS BASELINE
# ============================================================

comparison = pd.DataFrame({
    "Method": [
        "Rule-Based Baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_precision_at_50,
        model_precision_at_50
    ]
})

print("✅ Comparison table created.")

display(comparison)

13  Heading — Ranked Recommendations

In [ ]:
# ============================================================
# CAPSTONE STEP 11 — RANKED REVIEW QUEUE
# ============================================================

ranked_recommendations = test_results[
    [
        "content_hash_id",
        "client_hash_id",
        "decline_probability"
    ]
].copy()

ranked_recommendations["rank"] = (
    ranked_recommendations[
        "decline_probability"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

ranked_recommendations["action"] = np.select(
    [
        ranked_recommendations["decline_probability"] >= 0.80,
        ranked_recommendations["decline_probability"] >= 0.60,
        ranked_recommendations["decline_probability"] >= 0.40
    ],
    [
        "Review first",
        "Prioritize review",
        "Monitor"
    ],
    default="Protect"
)

ranked_recommendations = ranked_recommendations.sort_values(
    "rank"
)

print("✅ Ranked review queue created.")

display(
    ranked_recommendations.head(20)
)

14 Heading — Charts & Final Artifacts

In [ ]:
# ============================================================
# CAPSTONE STEP 12 — CHARTS AND FINAL ARTIFACTS
# ============================================================

# Chart 1 — Model vs Baseline

plt.figure(figsize=(7, 4))

plt.bar(
    comparison["Method"],
    comparison["Precision@50"]
)

plt.ylabel("Precision@50")
plt.title("Model vs Rule-Based Baseline")
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


# Chart 2 — Ranked decline probability

plt.figure(figsize=(8, 4))

plt.plot(
    ranked_recommendations["rank"].head(100),
    ranked_recommendations["decline_probability"].head(100)
)

plt.xlabel("Rank")
plt.ylabel("Decline Probability")
plt.title("Top 100 Ranked Content Review Opportunities")
plt.tight_layout()
plt.show()


# Final tables

print("MODEL COMPARISON")
display(comparison)

print("\nTOP 20 RECOMMENDATIONS")
display(
    ranked_recommendations.head(20)
)

print("\n✅ Charts and final tables generated.")

15 Heading — Save Artifacts

In [ ]:
# ============================================================
# CAPSTONE STEP 13 — SAVE FINAL ARTIFACTS
# ============================================================

comparison.to_csv(
    "capstone_model_comparison.csv",
    index=False
)

ranked_recommendations.head(100).to_csv(
    "capstone_ranked_review_queue.csv",
    index=False
)

print("✅ Artifacts saved:")
print("1. capstone_model_comparison.csv")
print("2. capstone_ranked_review_queue.csv")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

☑ Question filled
☑ Data filled
☑ Methodology filled
☑ Results filled
☑ Limitations filled
☑ Recommendations filled
☑ Charts generated
☑ Run All successful
☑ No private information
☑ Notebook saved to GitHub

# ML-12 — 5-Minute Demo Outline

## 1. Question — What problem am I solving? (45 seconds)

The goal of this project is to identify content that may be experiencing declining search performance and prioritize which pages should be reviewed first.

The main question is:

**Can a machine-learning model identify and rank potentially declining content more effectively than a simple rule-based baseline?**

The project is framed as decision support for content and SEO teams rather than as a prediction of Google's search algorithm.

## 2. Method — How did I approach it? (1 minute 30 seconds)

The workflow follows these steps:

1. Start with search-performance data.
2. Prepare and validate the data.
3. Create relevant features from the available search-performance signals.
4. Build a transparent rule-based baseline.
5. Train a machine-learning model.
6. Evaluate the model against the baseline.
7. Create a ranked content queue.
8. Convert the ranked results into practical recommendations.

The baseline provides a simple and explainable starting point, while the machine-learning model is evaluated to determine whether it provides a stronger ranking of pages for review.

## 3. One Chart — What should I show? (45 seconds)

I will show the main evaluation chart from the capstone notebook comparing the machine-learning approach with the rule-based baseline.

While presenting the chart, I will explain:

- what metric is being compared,
- why the metric matters for a ranked review queue,
- and how the model compares with the baseline.

The chart should be treated as evidence from this dataset and evaluation setup, not as a guarantee of future performance.

## 4. One Honest Result — What did the evaluation show? (45 seconds)

The evaluation showed that the machine-learning approach performed better than the rule-based baseline for the selected ranking task in the capstone evaluation.

The strongest result should be presented using the exact metric and value shown in the final notebook.

This result is directional and depends on the available data, selected features, and validation setup.

## 5. One Recommendation — What should a team do with the result? (45 seconds)

Use the model output as a prioritized review queue.

The highest-ranked content can be reviewed first by the relevant content or SEO team. Human review should then determine the appropriate action, such as checking content freshness, search intent alignment, page quality, or other relevant factors.

The model should support prioritization rather than automatically deciding what changes should be made.

## Closing — One Limitation (30 seconds)

The project uses anonymized search-performance data and should be treated as decision support.

The results are directional and may change with different data, features, validation designs, or environments.

The model does not predict Google's search algorithm and should not replace human review.

I built a machine-learning workflow to identify and prioritize content that may be experiencing declining search performance.

Instead of starting with a complex model, I first defined the content problem, prepared and validated the search-performance data, created a transparent rule-based baseline, and then compared it with a machine-learning approach.

The final workflow turns search-performance signals into a ranked review queue that can help content and SEO teams decide which pages to investigate first.

One important takeaway from the project was that evaluation matters as much as the model itself. I used the baseline as a reference point and treated the final result as directional decision support rather than a prediction of Google's search algorithm.

The project also reinforced the importance of human review: the model can help prioritize attention, while the actual content decision still requires human judgment.

#MachineLearning #DataScience #SEO #DataAnalytics #Python #FlyRank

## Social Post

## Employer-Facing Summary

I built a machine-learning pipeline using anonymized search-performance data to identify and prioritize content that may require review. The workflow included data preparation, feature engineering, a rule-based baseline, machine-learning modeling, evaluation, and generation of a ranked content queue. The evaluation showed that the machine-learning approach provided stronger ranking performance than the baseline for the selected task, while the result remained directional and intended to support human review.